In [2]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Anish494/flyrank_ai_first_assignment"
REPO_DIR = "flyrank_ai_first_assignment"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
print(f"{len(df):,} rows ready")

30,000 rows ready


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anish494/flyrank_ai_first_assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Signal 1 — Staleness (ties to the `stale_visible_page` flag).**
Verdict: **OPPOSITE**. Non-stale pages decline *more* (54.2% "down", n=29,826) than
stale pages (47.1% "down", n=174). This directly contradicts the assumption behind
the staleness flag. Caveat: the stale-page sample is tiny (n=174 vs 29,826), so this
result should be treated as a red flag against blindly using staleness, not as strong
proof staleness is protective.

**Signal 2 — Volume (ties to quick-win-style volume logic).**
Verdict: **MIXED**. Decline rate rises from low (38.9%, n=8,006) to medium (60.3%,
n=8,485) to high (62.0%, n=9,907), but then drops back down at very_high (52.4%,
n=3,602). The relationship is real but not monotonic, so I won't lean on volume alone
as a clean single signal.

**My rule (given both checks came back OPPOSITE/MIXED, not CONFIRMED):**
Rather than build a rule on staleness (which the data contradicts) or ranking purely by
raw volume (non-monotonic), I'll use a rule closer to the lane guide's
`declining_with_demand` pattern: flag pages that are BOTH currently marked "down" AND
have enough visible demand (impressions_90d >= 500) to make a review worthwhile. This
uses `trend_direction` (an observed signal, not future-derived) and a volume floor,
avoiding the misleading staleness signal entirely.

Score: `impressions_90d` for pages where `trend_direction == "down"` and
`impressions_90d >= 500`; zero otherwise.
Reason code: `declining_with_demand`
Action label: `review_for_refresh`

In [3]:
# Signal check 1: staleness -- is a stale page (not updated in 180+ days)
# actually associated with declining trend? (ties to the stale_visible_page flag)
df["is_stale"] = df["days_since_last_update"] >= 180
stale_bucket = df.groupby("is_stale")["trend_direction"].value_counts(normalize=True).unstack().round(3)
stale_n = df.groupby("is_stale").size()
print("Staleness vs trend_direction (row proportions):")
print(stale_bucket)
print("\nn per bucket:")
print(stale_n)

# Signal check 2: volume -- does high impressions_90d associate with a specific
# trend pattern? (ties to quick-win style volume logic)
df["volume_tier"] = pd.cut(df["impressions_90d"], bins=[0, 100, 1000, 10000, np.inf],
                             labels=["low", "medium", "high", "very_high"])
volume_bucket = df.groupby("volume_tier", observed=True)["trend_direction"].value_counts(normalize=True).unstack().round(3)
volume_n = df.groupby("volume_tier", observed=True).size()
print("\n\nVolume tier vs trend_direction (row proportions):")
print(volume_bucket)
print("\nn per bucket:")
print(volume_n)

Staleness vs trend_direction (row proportions):
trend_direction   down   flat    new  stable     up
is_stale                                           
False            0.542  0.038  0.074   0.199  0.146
True             0.471  0.092  0.144   0.138  0.155

n per bucket:
is_stale
False    29826
True       174
dtype: int64


Volume tier vs trend_direction (row proportions):
trend_direction   down   flat    new  stable     up
volume_tier                                        
low              0.389  0.139  0.253   0.092  0.127
medium           0.603  0.004  0.016   0.185  0.193
high             0.620  0.000  0.007   0.247  0.125
very_high        0.524    NaN  0.002   0.337  0.137

n per bucket:
volume_tier
low          8006
medium       8485
high         9907
very_high    3602
dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import os

# Encode the rule from Section 1
df["baseline_score"] = np.where(
    (df["trend_direction"] == "down") & (df["impressions_90d"] >= 500),
    df["impressions_90d"],
    0
)
df["reason_code"] = np.where(df["baseline_score"] > 0, "declining_with_demand", "no_action")
df["action_label"] = np.where(df["baseline_score"] > 0, "review_for_refresh", "monitor")

# Build the ranked queue
queue = df.sort_values("baseline_score", ascending=False)[
    ["content_id", "baseline_score", "reason_code", "action_label",
     "impressions_90d", "trend_direction", "avg_position", "ctr", "content_type"]
].reset_index(drop=True)

print(f"{(queue['baseline_score'] > 0).sum():,} pages flagged for review out of {len(queue):,} total")
queue.head(10)

# Write the CSV (stays out of git by design -- CI leak-guard blocks data files)
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("\nWritten to work/outputs/baseline_action_score.csv")

9,961 pages flagged for review out of 30,000 total

Written to work/outputs/baseline_action_score.csv


## 3. Top-10 review

1. **content_5fe46e04994d** — review_for_refresh. Highest score (517,715 impressions),
   declining, strong position (4.2), reasonable CTR (0.14). What would make this wrong:
   if this decline is seasonal/temporary rather than a real structural drop — the rule
   can't tell the difference (per the lane guide's decline-vs-seasonality warning).
2. **content_8c19996aa890** — review_for_refresh. Very similar profile to #1: huge
   volume, strong position (2.5), declining. What would make it wrong: same
   seasonality risk, plus this could be a case of consolidation if a sibling page
   absorbed its traffic rather than a genuine drop.
3. **content_4c36c775b818** — review_for_refresh. Good position (2.3) and decent CTR
   (0.41) despite declining. What would make it wrong: a page performing this well on
   CTR and position might just be having a temporary dip, not a real refresh candidate.
4. **content_1a9e894be2e2** — review_for_refresh. Strong position (4.0), moderate CTR
   (0.23). What would make it wrong: same as above — strong underlying metrics muddy
   the "declining = needs refresh" story.
5. **content_2c2606c5d176** — review_for_refresh. Best CTR so far (0.53) with strong
   position (4.2). What would make it wrong: a page with CTR this high is arguably
   *healthy*, not a refresh priority — my rule doesn't account for CTR at all, only
   volume and decline flag, so it may be over-flagging genuinely fine pages.
6. **content_cb112fce36be** — review_for_refresh. Position 5.6, CTR 0.16 — unremarkable,
   fits the intended pattern reasonably well.
7. **content_9532f197bbc8** — review_for_refresh. CTR of 0.87 is unusually high for a
   "declining" page — this is suspicious. What would make it wrong: this might be a
   data quality issue (very low impression count making CTR noisy) or a
   mislabeled/edge-case row that shouldn't be flagged as a refresh priority at all.
8. **content_008fb02c46cb** — review_for_refresh. Position 4.4, fits the pattern.
9. **content_813e88069237** — review_for_refresh. Position jumps to 26.2 (page 3) with
   low CTR (0.06) — a genuinely different profile from rows 1-8, which mostly sat in
   the top 5 positions. What would make it wrong: nothing, actually — this looks like
   a legitimately strong candidate: high volume, poor position, poor CTR, declining.
10. **content_ff94c9b6b411** — review_for_refresh. Similar to #9: weak position (27.4),
    very low CTR (0.04), declining. What would make it wrong: this is arguably my
    *most* confident pick in the top 10 — every signal points the same direction.

**Overall observation:** all 10 top rows are `content_type == "keyword article"` —
my rule is entirely volume-driven, so it's implicitly dominated by whichever content
type has the highest-traffic pages. This is a real limitation: the rule isn't
comparing pages fairly across content types, it's just surfacing whoever has the most
raw traffic. Rows 1-6 also have strong CTR/position despite being flagged "declining,"
which is a mismatch worth investigating — a truly weak page (like #9 and #10) is a
much more convincing candidate than a strong-performing page that happens to have
dipped slightly.

In [5]:
queue.head(10)

,content_id,baseline_score,reason_code,action_label,impressions_90d,trend_direction,avg_position,ctr,content_type
0,content_5fe46e04994d,517715,declining_with_demand,review_for_refresh,517715,down,4.2,0.14,keyword article
1,content_8c19996aa890,509252,declining_with_demand,review_for_refresh,509252,down,2.5,0.15,keyword article
2,content_4c36c775b818,463103,declining_with_demand,review_for_refresh,463103,down,2.3,0.41,keyword article
3,content_1a9e894be2e2,416180,declining_with_demand,review_for_refresh,416180,down,4.0,0.23,keyword article
4,content_2c2606c5d176,347399,declining_with_demand,review_for_refresh,347399,down,4.2,0.53,keyword article
5,content_cb112fce36be,309910,declining_with_demand,review_for_refresh,309910,down,5.6,0.16,keyword article
6,content_9532f197bbc8,309192,declining_with_demand,review_for_refresh,309192,down,2.0,0.87,keyword article
7,content_008fb02c46cb,236803,declining_with_demand,review_for_refresh,236803,down,4.4,0.26,keyword article
8,content_813e88069237,233561,declining_with_demand,review_for_refresh,233561,down,26.2,0.06,keyword article
9,content_ff94c9b6b411,228566,declining_with_demand,review_for_refresh,228566,down,27.4,0.04,keyword article


## 4. Weak picks + leakage check

**Weakest picks in my top 10:**

- **#5 (content_2c2606c5d176)** — CTR of 0.53 is strong, and combined with a good
  position (4.2), this page doesn't look like it's actually struggling. My rule flagged
  it purely because it's high-volume and technically marked "down," but the underlying
  health signals (CTR, position) suggest this may be a false positive — possibly a
  small, temporary dip on an otherwise healthy page rather than a real refresh need.
- **#7 (content_9532f197bbc8)** — the CTR of 0.87 is unusually high and worth
  double-checking before trusting this row at all; it may reflect a low-impression
  edge case where CTR is noisy rather than a meaningful signal, even though this page
  passed my ≥500 impression filter.

**Strongest picks, by contrast:** #9 and #10 combine weak position, weak CTR, high
volume, AND a declining trend — every signal agrees, unlike the picks above where only
volume and the trend flag agree while CTR/position suggest otherwise. This gap between
"technically flagged" and "genuinely convincing" is exactly why the top-10 read matters:
the rule's raw ranking isn't identical to which pages actually deserve review first.

**Leakage check:**
- The rule uses only `trend_direction` and `impressions_90d` — both are observed,
  present-window signals, not future-window or label-derived features.
- `trend_direction` is itself a bucketed/derived field (from `trend_pct`), so it is
  already a mild proxy rather than a raw measurement — but per Section 3 of this
  notebook and the ML-04 leak-trap exercise, this is the same starter label used
  throughout the track (`is_declining_label = trend_direction == "down"`), and I have
  not fed in `trend_pct` itself or any other future-window metric.
- No product decision flags (`health_score`, `priority_score`, `action_type`) were
  used anywhere in this rule — confirmed by checking Section 1's feature list against
  the "Product context" field-type table in the lane guide.
- Confirmed: no future-window or label-derived inputs beyond the standard starter
  `trend_direction` proxy that every baseline in this track is built on.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.